# DAH 2026 해금팀 — UGV 지상 이상탐지 VAE
**담당:** 광빈 | **환경:** Google Colab + PyTorch  
**탐지 대상:** GPS Spoofing / Wheel Slip / Command Anomaly  
**XAI:** 피처별 재구성 오차 기여도 분석

In [ ]:
# Step 0. 라이브러리 설치
!pip install shap -q

In [ ]:
# Step 1. 라이브러리 임포트
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from torch.utils.data import DataLoader, TensorDataset

# GPS 속도 + GPS-IMU residual 포함 (10개 피처)
# residual = GPS속도 - IMU가속도 → 정상이면 ~0, 공격받으면 폭발
FEATURE_NAMES = ['gps_vel_x', 'gps_vel_y',
                 'imu_ax',    'imu_ay',
                 'residual_x','residual_y',
                 'wheel_vel_l','wheel_vel_r',
                 'cmd_vel_x', 'cmd_vel_w']
print('임포트 완료')

## Step 2. 합성 데이터 생성
실제 UGV 센서 데이터가 없으므로 시뮬레이션으로 생성합니다.  
**실제 GPS Spoofing 데이터:** Mendeley Data (DOI: 10.17632/z7dj3yyzt8.3)로 추후 교체 가능

In [ ]:
def generate_normal(n=1000):
    """정상 UGV 직선 주행 데이터"""
    t = np.linspace(0, 10, n)
    gps_x = t * 0.5 + np.random.normal(0, 0.01, n)
    gps_y = np.sin(t * 0.1) * 0.3 + np.random.normal(0, 0.01, n)
    gps_vel_x = np.gradient(gps_x)
    gps_vel_y = np.gradient(gps_y)
    imu_ax = gps_vel_x + np.random.normal(0, 0.005, n)
    imu_ay = gps_vel_y + np.random.normal(0, 0.005, n)
    residual_x = gps_vel_x - imu_ax   # 정상: ≈ 0
    residual_y = gps_vel_y - imu_ay
    wheel_l   = 0.5 + np.random.normal(0, 0.01, n)
    wheel_r   = 0.5 + np.random.normal(0, 0.01, n)
    cmd_vel_x = 0.5 + np.random.normal(0, 0.01, n)
    cmd_vel_w = 0.0 + np.random.normal(0, 0.005, n)
    return np.stack([gps_vel_x, gps_vel_y, imu_ax, imu_ay,
                     residual_x, residual_y,
                     wheel_l, wheel_r, cmd_vel_x, cmd_vel_w], axis=1)

def generate_gps_spoofing(n=300):
    """GPS Spoofing: 가짜 GPS 신호로 gps_vel 순간 폭발, IMU는 정상 → residual 폭발"""
    data = generate_normal(n)
    jump = n // 3
    data[jump:jump+15, 0] += 8.0   # gps_vel_x 폭발
    data[jump:jump+15, 1] += 4.0   # gps_vel_y 폭발
    data[jump:jump+15, 4] += 8.0   # residual_x 폭발 (GPS-IMU 불일치)
    data[jump:jump+15, 5] += 4.0   # residual_y 폭발
    return data

def generate_wheel_slip(n=300):
    """Wheel Slip: 앞 1/3 정상 주행, 이후 바퀴 급회전 but IMU≈0 (실제 이동 없음)"""
    data = generate_normal(n)
    start = n // 3   # 100번 샘플 이후부터 이상
    data[start:, 6] += np.random.normal(0.8, 0.1, n - start)
    data[start:, 7] += np.random.normal(0.8, 0.1, n - start)
    data[start:, 2]  = np.random.normal(0, 0.005, n - start)
    data[start:, 3]  = np.random.normal(0, 0.005, n - start)
    return data

def generate_command_anomaly(n=300):
    """Command Anomaly: 앞 1/3 정상, 이후 cmd_vel 정상이지만 바퀴 무반응"""
    data = generate_normal(n)
    start = n // 3   # 100번 샘플 이후부터 이상
    data[start:, 8] = 0.5 + np.random.normal(0, 0.01, n - start)
    data[start:, 6] = np.random.normal(0, 0.02, n - start)
    data[start:, 7] = np.random.normal(0, 0.02, n - start)
    return data

def sliding_window(data, window=20):
    return np.array([data[i:i+window] for i in range(len(data) - window)])

print('데이터 함수 정의 완료')

## Step 3. VAE 모델 정의

In [ ]:
class VAE(nn.Module):
    def __init__(self, input_dim, latent_dim=16):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(),
            nn.Linear(128, 64),        nn.ReLU()
        )
        self.fc_mu     = nn.Linear(64, latent_dim)
        self.fc_logvar = nn.Linear(64, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64), nn.ReLU(),
            nn.Linear(64, 128),        nn.ReLU(),
            nn.Linear(128, input_dim)
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + std * torch.randn_like(std)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decoder(z), mu, logvar

def vae_loss(recon, x, mu, logvar, beta=1.0):
    recon_loss = nn.MSELoss()(recon, x)
    kl_loss    = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + beta * kl_loss

print('VAE 모델 정의 완료')

## Step 4. 데이터 준비 + 학습

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
WINDOW = 20
N_FEAT = len(FEATURE_NAMES)  # 10

normal   = generate_normal(1000)
spoof    = generate_gps_spoofing(300)
slip     = generate_wheel_slip(300)
cmd_anom = generate_command_anomaly(300)

mean, std = normal.mean(0), normal.std(0) + 1e-8
def prep(x):
    normed = (x - mean) / std
    windows = sliding_window(normed, WINDOW)
    return windows.reshape(-1, WINDOW * N_FEAT)

X_normal = prep(normal)
X_spoof  = prep(spoof)
X_slip   = prep(slip)
X_cmd    = prep(cmd_anom)

split   = int(len(X_normal) * 0.8)
X_train = torch.FloatTensor(X_normal[:split])
X_val   = X_normal[split:]
loader  = DataLoader(TensorDataset(X_train), batch_size=32, shuffle=True)

model     = VAE(input_dim=WINDOW * N_FEAT, latent_dim=16).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

history = []
for epoch in range(60):
    model.train()
    total = 0
    for (batch,) in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        recon, mu, logvar = model(batch)
        loss = vae_loss(recon, batch, mu, logvar)
        loss.backward()
        optimizer.step()
        total += loss.item()
    avg = total / len(loader)
    history.append(avg)
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1:3d}/60 | Loss: {avg:.4f}')

print('\n학습 완료!')

## Step 5. 이상 점수 계산 + 탐지율

In [ ]:
def anomaly_score(model, X, device='cpu'):
    model.eval()
    with torch.no_grad():
        x = torch.FloatTensor(X).to(device)
        recon, _, _ = model(x)
        return torch.mean((recon - x) ** 2, dim=1).cpu().numpy()

s_normal = anomaly_score(model, X_val,   device)
s_spoof  = anomaly_score(model, X_spoof, device)
s_slip   = anomaly_score(model, X_slip,  device)
s_cmd    = anomaly_score(model, X_cmd,   device)

threshold = np.percentile(s_normal, 95)
print(f'임계값 (정상 95th): {threshold:.4f}')
print(f'GPS Spoofing  탐지율: {(s_spoof > threshold).mean()*100:.1f}%')
print(f'Wheel Slip    탐지율: {(s_slip  > threshold).mean()*100:.1f}%')
print(f'Cmd Anomaly   탐지율: {(s_cmd   > threshold).mean()*100:.1f}%')

## Step 6. 탐지 결과 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history, color='steelblue')
axes[0].set_title('VAE 학습 Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')

axes[1].boxplot([s_normal, s_spoof, s_slip, s_cmd],
                labels=['정상', 'GPS\nSpoofing', 'Wheel\nSlip', 'Cmd\nAnomaly'])
axes[1].axhline(threshold, color='red', linestyle='--',
                label=f'임계값={threshold:.3f}')
axes[1].set_title('공격 유형별 이상 점수')
axes[1].set_ylabel('Anomaly Score (재구성 오차)')
axes[1].legend()

plt.tight_layout()
plt.savefig('ugv_anomaly_result.png', dpi=150)
plt.show()
print('완료!')

## Step 7. XAI — 피처별 기여도 분석
"왜 이 창에서 이상이 감지됐는가" — 어떤 센서가 원인인지 설명  
재구성 오차를 피처별로 분리하여 시각화

In [ ]:
def feature_contribution(model, X, device='cpu'):
    """피처별 재구성 오차 → 어떤 센서가 이상 점수를 높였는지 설명"""
    model.eval()
    with torch.no_grad():
        x = torch.FloatTensor(X).to(device)
        recon, _, _ = model(x)
        per_elem = (recon - x) ** 2
        # (batch, WINDOW*N_FEAT) → (batch, WINDOW, N_FEAT)
        per_elem = per_elem.reshape(-1, WINDOW, N_FEAT)
        return per_elem.mean(dim=1).cpu().numpy()  # 시간축 평균 → (batch, N_FEAT)

# 각 공격 유형에서 이상 점수 최고 샘플 선택
idx_spoof = np.argmax(s_spoof)
idx_slip  = np.argmax(s_slip)
idx_cmd   = np.argmax(s_cmd)

contrib_spoof = feature_contribution(model, X_spoof[idx_spoof:idx_spoof+1], device)[0]
contrib_slip  = feature_contribution(model, X_slip[idx_slip:idx_slip+1],   device)[0]
contrib_cmd   = feature_contribution(model, X_cmd[idx_cmd:idx_cmd+1],      device)[0]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

plot_data = [
    (contrib_spoof, 'GPS Spoofing',      'crimson'),
    (contrib_slip,  'Wheel Slip',         'darkorange'),
    (contrib_cmd,   'Command Anomaly',    'steelblue'),
]

for ax, (contrib, title, color) in zip(axes, plot_data):
    sorted_idx = np.argsort(contrib)[::-1]
    ax.barh(
        [FEATURE_NAMES[i] for i in sorted_idx],
        contrib[sorted_idx],
        color=color, alpha=0.85
    )
    ax.set_xlabel('재구성 오차 기여도')
    ax.set_title(f'{title}\n이상 원인 분석')
    ax.invert_yaxis()

plt.suptitle('VAE 이상탐지 XAI — 피처별 기여도\n(높을수록 해당 센서가 이상의 주요 원인)', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('ugv_xai_contribution.png', dpi=150)
plt.show()
print('XAI 분석 완료! ugv_xai_contribution.png 저장됨')

## Step 8. 결과 요약표 (보고서 6.6절 용)
탐지 성능 정량 평가

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

y_true_spoof = np.ones(len(s_spoof))
y_true_slip  = np.ones(len(s_slip))
y_true_cmd   = np.ones(len(s_cmd))

y_pred_spoof = (s_spoof > threshold).astype(int)
y_pred_slip  = (s_slip  > threshold).astype(int)
y_pred_cmd   = (s_cmd   > threshold).astype(int)

print('=' * 55)
print(f'{'공격 유형':<18} {'Precision':>10} {'Recall':>8} {'F1':>8}')
print('-' * 55)
for name, yt, yp in [
    ('GPS Spoofing',    y_true_spoof, y_pred_spoof),
    ('Wheel Slip',      y_true_slip,  y_pred_slip),
    ('Command Anomaly', y_true_cmd,   y_pred_cmd),
]:
    p = precision_score(yt, yp, zero_division=0)
    r = recall_score(yt, yp)
    f = f1_score(yt, yp, zero_division=0)
    print(f'{name:<18} {p:>10.3f} {r:>8.3f} {f:>8.3f}')
print('=' * 55)
print(f'\n임계값: {threshold:.4f} (정상 데이터 95th 퍼센타일)')
print(f'모델: VAE (latent_dim=16, window=20, feature=10)')

In [ ]:
## Step 8-2. 혼동행렬 (Confusion Matrix)
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 4, figsize=(20, 4))

# 공격 유형별 (정상 + 각 공격 혼합)
configs = [
    ('전체 (All)',      np.concatenate([s_spoof, s_slip, s_cmd])),
    ('GPS Spoofing',   s_spoof),
    ('Wheel Slip',     s_slip),
    ('Cmd Anomaly',    s_cmd),
]

for ax, (name, s_atk) in zip(axes, configs):
    y_true = np.concatenate([np.zeros(len(s_normal)), np.ones(len(s_atk))])
    y_pred = (np.concatenate([s_normal, s_atk]) > threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['정상', '이상'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontweight='bold')

plt.suptitle('혼동행렬 — VAE 이상탐지 분류 성능', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('ugv_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('저장: ugv_confusion_matrix.png')

In [ ]:
## Step 9. 가중치 저장 (anomaly_detector.py 연동용)
# anomaly_detector.py가 'vae_ugv.pth'를 로드함
torch.save(model.state_dict(), 'vae_ugv.pth')
print('✅ vae_ugv.pth 저장 완료 — anomaly_detector.py (platform=ugv) 에서 사용')

# 검증: 저장된 가중치 다시 로드해서 동일 결과 확인
model_loaded = VAE(input_dim=WINDOW * N_FEAT, latent_dim=16)
model_loaded.load_state_dict(torch.load('vae_ugv.pth', map_location='cpu'))
model_loaded.eval()
s_check = anomaly_score(model_loaded, X_val, 'cpu')
print(f'재로드 검증 — 임계값: {np.percentile(s_check, 95):.4f} (원본과 동일해야 함)')

## Step 10. 시계열 이상 점수 — 공격 발생 시점 시각화
공격이 시작되는 순간 이상 점수가 급등하는 것을 시간 축으로 보여줌  
→ 발표용 핵심 그래프: "VAE가 GPS Spoofing을 즉시 탐지했다"

In [ ]:
def timeseries_score(model, data, mean, std, window=20, n_feat=10, device='cpu'):
    """원본 시계열 데이터 → 각 타임스텝의 이상 점수"""
    normed = (data - mean) / std
    X = sliding_window(normed, window).reshape(-1, window * n_feat)
    return anomaly_score(model, X, device)

def detect_latency(scores, threshold, attack_start_score, timestep_ms=100):
    """공격 시작(스코어 배열 기준) 이후 첫 탐지까지의 지연 시간"""
    after = scores[attack_start_score:]
    detected = np.where(after > threshold)[0]
    if len(detected) == 0:
        return None, None
    steps = int(detected[0])
    return steps, steps * timestep_ms

ATTACK_START_SAMPLE = 100   # generate_*() 공통: n//3 = 100
ATTACK_START_SCORE  = ATTACK_START_SAMPLE - WINDOW   # 슬라이딩 윈도우 오프셋 = 80

fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=False)

attack_configs = [
    (generate_gps_spoofing(300),    'GPS Spoofing',    'crimson',
     [(100, 115, 'GPS Spoofing 시작')]),
    (generate_wheel_slip(300),      'Wheel Slip',       'darkorange',
     [(100, 300, 'Wheel Slip 시작')]),
    (generate_command_anomaly(300), 'Command Anomaly',  'steelblue',
     [(100, 300, 'Cmd Anomaly 시작')]),
]

for ax, (atk_data, title, color, attack_regions) in zip(axes, attack_configs):
    scores = timeseries_score(model, atk_data, mean, std, WINDOW, N_FEAT, device)
    t = np.arange(len(scores))

    ax.plot(t, scores, color=color, linewidth=1.2, label='이상 점수')
    ax.axhline(threshold, color='black', linestyle='--',
               linewidth=1.2, label=f'임계값 {threshold:.3f}')
    ax.fill_between(t, 0, scores,
                    where=(scores > threshold),
                    alpha=0.3, color=color, label='탐지 구간')

    for (x0, x1, label) in attack_regions:
        score_x0 = max(0, x0 - WINDOW)
        score_x1 = min(len(scores), x1 - WINDOW)
        ax.axvspan(score_x0, score_x1, alpha=0.12, color='gray')
        ax.text(score_x0 + 2, scores.max() * 0.75,
                label, fontsize=9, color='dimgray')

    # 탐지 지연 시간 계산 + 표시
    latency_steps, latency_ms = detect_latency(scores, threshold, ATTACK_START_SCORE)
    if latency_steps is not None:
        detect_x = ATTACK_START_SCORE + latency_steps
        ax.annotate(
            f'탐지 지연: {latency_steps}스텝 ({latency_ms}ms)',
            xy=(detect_x, threshold),
            xytext=(detect_x + 10, scores.max() * 0.55),
            fontsize=9, color='darkred', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='darkred', lw=1.2),
        )
        print(f'[{title}] 탐지 지연 = {latency_steps}스텝 ({latency_ms}ms)'
              f' | 공격 시작 스텝: {ATTACK_START_SCORE}')
    else:
        print(f'[{title}] 미탐지')

    ax.set_title(f'{title} — 시계열 이상 점수', fontweight='bold')
    ax.set_ylabel('Anomaly Score')
    ax.set_xlabel('타임스텝 (슬라이딩 윈도우 기준, 1스텝=100ms)')
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('UGV VAE 이상탐지 — 공격 발생 시점 시각화\n'
             '(임계값 초과 = 탐지, 회색 영역 = 실제 공격 구간)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('ugv_timeseries_detection.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n저장: ugv_timeseries_detection.png')